In [11]:
pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/317.2 MB ? eta -:--:--
     --------------------------------------- 1.8/317.2 MB 14.3 MB/s eta 0:00:23
      -------------------------------------- 6.8/317.2 MB 20.0 MB/s eta 0:00:16
     - ------------------------------------ 12.1/317.2 MB 22.2 MB/s eta 0:00:14
     - ------------------------------------ 16.0/317.2 MB 22.4 MB/s eta 0:00:14
     -- ----------------------------------- 22.5/317.2 MB 23.0 MB/s eta 0:00:13
     --- ---------------------------------- 28.8/317.2 MB 24.1 MB/s eta 0:00:12
     ---- --------------------------------- 35.4/317.2 MB 25.0 MB/s eta 0:00:12
     ---- --------------------------------- 41.7/317.2 MB 25.7 MB/s eta 0:00:11
     ----- -------------------------------- 48.0/317.2 MB 25.9 MB/s eta 0:00:11
     ------ ------------------------------- 54.0/317.2 MB 26.3 MB/s eta 0:00:11
     ------- ------------------------------ 60.3/

In [1]:
import os
import urllib.request

# Paso 1: Definir ruta destino
hadoop_home = "C:\\hadoop"
bin_dir = os.path.join(hadoop_home, "bin")
winutils_url = "https://github.com/steveloughran/winutils/raw/master/hadoop-2.7.1/bin/winutils.exe"
winutils_path = os.path.join(bin_dir, "winutils.exe")

# Paso 2: Crear carpeta si no existe
os.makedirs(bin_dir, exist_ok=True)

# Paso 3: Descargar winutils.exe si no está
if not os.path.isfile(winutils_path):
    print("Descargando winutils.exe...")
    urllib.request.urlretrieve(winutils_url, winutils_path)
    print("Descarga completa.")
else:
    print("winutils.exe ya existe. No se descarga de nuevo.")

# Paso 4: Establecer variable de entorno
os.environ["HADOOP_HOME"] = hadoop_home
print("HADOOP_HOME configurado correctamente.")

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("HeartDiseaseBatchProcessing") \
    .getOrCreate()

# Cargar desde archivo CSV
df = spark.read.csv("C:\\Users\\Juanfe\\Downloads\\heart_disease_uci.csv", header=True, inferSchema=True)

winutils.exe ya existe. No se descarga de nuevo.
HADOOP_HOME configurado correctamente.


In [3]:
# Revisar tipos de datos y valores nulos
df.printSchema()
df.show(5)
df.describe().show()

# Eliminar registros con valores nulos
df_clean = df.dropna()

# Transformar valores categóricos si es necesario
from pyspark.sql.functions import col, when

df_transformed = df_clean.withColumn(
    "sex", when(col("sex") == 1, "Male").otherwise("Female")
)

root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- dataset: string (nullable = true)
 |-- cp: string (nullable = true)
 |-- trestbps: integer (nullable = true)
 |-- chol: integer (nullable = true)
 |-- fbs: boolean (nullable = true)
 |-- restecg: string (nullable = true)
 |-- thalch: integer (nullable = true)
 |-- exang: boolean (nullable = true)
 |-- oldpeak: double (nullable = true)
 |-- slope: string (nullable = true)
 |-- ca: integer (nullable = true)
 |-- thal: string (nullable = true)
 |-- num: integer (nullable = true)

+---+---+------+---------+---------------+--------+----+-----+--------------+------+-----+-------+-----------+---+-----------------+---+
| id|age|   sex|  dataset|             cp|trestbps|chol|  fbs|       restecg|thalch|exang|oldpeak|      slope| ca|             thal|num|
+---+---+------+---------+---------------+--------+----+-----+--------------+------+-----+-------+-----------+---+--------------

In [10]:
spark.conf.set("spark.hadoop.fs.defaultFS", "file:///")

# Conteo por género
df_transformed.groupBy("sex").count().show()

# Estadísticas por grupo
df_transformed.groupBy("num").agg({"age": "avg", "chol": "avg"}).show()

+------+-----+
|   sex|count|
+------+-----+
|Female|  299|
+------+-----+

+---+------------------+------------------+
|num|         avg(chol)|          avg(age)|
+---+------------------+------------------+
|  1|246.07142857142858|55.464285714285715|
|  3|246.45714285714286|              56.0|
|  4| 253.3846153846154| 59.69230769230769|
|  2|260.85714285714283|              58.2|
|  0|         243.49375|          52.64375|
+---+------------------+------------------+



In [32]:
df_transformed.toPandas().to_csv("C:\\Users\\Juanfe\\Downloads\\resultados\\heart_processed.csv", index=False)